# Case Study: Cybersecurity “Suspicious Login” Alerts (Precision vs Recall)

In this notebook, we build a classifier that flags **suspicious logins** (possible account takeover) and use it to answer:

- When should we focus on **recall**?
- When should we focus on **precision**?
- How does the **decision threshold** change the trade-off?
- How can we choose a threshold using a simple **cost-based** rule?

## Definitions (Binary Classification)

- **Positive class (1)**: suspicious / compromised login  
- **Negative class (0)**: legitimate login  

### Confusion Matrix (for class “1”)
| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| **Actual 0** | TN | FP |
| **Actual 1** | FN | TP |

### Key interpretation for this case
- **False Negative (FN)**: a real attack is classified as benign → attacker slips through (**potentially severe** impact: breach/ransomware)
- **False Positive (FP)**: a legitimate login is flagged → user friction, helpdesk tickets

> **Rule of thumb:** When **FN cost is high**, prioritize **recall**. When **FP cost is high**, prioritize **precision**.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    classification_report,
    precision_recall_curve
)
df = pd.read_csv('dataset/login_risk_1000.xls')
df.head()

,failed_logins,geo_distance_km,new_device,login_hour,impossible_travel,attack
0,2,11,0,14,0,0
1,6,2099,1,23,1,1
2,0,79,1,15,0,0
3,1,11,0,16,0,0
4,0,25,0,16,0,0


## 1) “login risk” dataset


- `failed_logins`: number of failed attempts in last 10 min  
- `geo_distance_km`: distance from last known login location  
- `new_device`: 1 if device not seen before  
- `login_hour`: hour of day (0–23)
- `impossible travel`: is when a user appears to log in from two far-apart locations within a time span that a human could not realistically travel between.
- `attack`: is a login attempt that is flagged as malicious or unauthorized.

“attacks” **rare** (class imbalance), which is common in real security problems.


## 2) Train a baseline classifier

We’ll use **Logistic Regression** and keep the output as **probabilities** so we can adjust thresholds later.


In [9]:
X = df[
    [
        "failed_logins",
        "geo_distance_km",
        "new_device",
        "login_hour",
        "impossible_travel",
    ]
]

y=df["attack"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]

## 3) Helper: evaluate at a chosen decision threshold

By default, many classifiers use **threshold = 0.5**:
- Predict **1** if `P(class=1) >= 0.5`
- Otherwise predict **0**

But **you** can choose thresholds based on the real-world costs of FP vs FN.


In [10]:
def evaluate_at_threshold(y_true, y_prob, threshold=0.5):
    """Return confusion matrix and precision/recall for a given threshold."""
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)  # [[TN, FP], [FN, TP]]
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return cm, prec, rec


## 4) Baseline evaluation (threshold = 0.5)

Interpretation reminders:
- **Precision**: among predicted attacks, what fraction are truly attacks?
- **Recall**: among true attacks, what fraction did we catch?


In [11]:
cm_05, p_05, r_05 = evaluate_at_threshold(y_test, proba, threshold=0.5)

print("Threshold = 0.50")
print("Confusion Matrix [[TN, FP], [FN, TP]]:\n", cm_05)
print(f"Precision: {p_05:.3f}")
print(f"Recall:    {r_05:.3f}\n")

print("Classification report (threshold=0.5):")
y_pred_05 = (proba >= 0.5).astype(int)
print(classification_report(y_test, y_pred_05, digits=3, zero_division=0))


Threshold = 0.50
Confusion Matrix [[TN, FP], [FN, TP]]:
 [[216   1]
 [ 20  13]]
Precision: 0.929
Recall:    0.394

Classification report (threshold=0.5):
              precision    recall  f1-score   support

           0      0.915     0.995     0.954       217
           1      0.929     0.394     0.553        33

    accuracy                          0.916       250
   macro avg      0.922     0.695     0.753       250
weighted avg      0.917     0.916     0.901       250



## 5) Precision–Recall trade-off by changing threshold

Lower threshold → more things flagged → **recall tends to increase** (fewer FN), but precision may drop (more FP).  
Higher threshold → fewer things flagged → **precision tends to increase**, but recall may drop (more FN).

Run the loop and compare FP vs FN counts.


In [12]:
for t in [0.85, 0.70, 0.50, 0.35, 0.20, 0.10]:
    cm, prec, rec = evaluate_at_threshold(y_test, proba, threshold=t)
    tn, fp, fn, tp = cm.ravel()
    print(f"Threshold={t:.2f} | Precision={prec:.3f} Recall={rec:.3f} | FP={fp} FN={fn} TP={tp}")


Threshold=0.85 | Precision=1.000 Recall=0.303 | FP=0 FN=23 TP=10
Threshold=0.70 | Precision=0.923 Recall=0.364 | FP=1 FN=21 TP=12
Threshold=0.50 | Precision=0.929 Recall=0.394 | FP=1 FN=20 TP=13
Threshold=0.35 | Precision=0.929 Recall=0.394 | FP=1 FN=20 TP=13
Threshold=0.20 | Precision=0.929 Recall=0.394 | FP=1 FN=20 TP=13
Threshold=0.10 | Precision=0.500 Recall=0.485 | FP=16 FN=17 TP=16


### Class discussion (1–2 minutes)

1. Which threshold feels “**security-first**” (low FN)?  
2. Which threshold feels “**user-friction-first**” (low FP)?  


## 6) Cost-based threshold selection (decision framework)

A practical way to choose a threshold is to assign “costs”:

- **Cost(FN)**: missing a real attack (often very high)
- **Cost(FP)**: disrupting a legit user (usually lower, but not always)

Then choose the threshold that minimizes expected cost:

$$
\text{Cost} = FP \cdot c_{FP} + FN \cdot c_{FN}
$$
Try the default costs below, then change them and re-run.


In [13]:
def expected_cost(y_true, y_prob, threshold, cost_fp=20, cost_fn=5000):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * cost_fp + fn * cost_fn

# Example costs (experiement with other values)
C_FP = 20     # e.g., helpdesk time / user frustration
C_FN = 5000   # e.g., breach risk / incident response cost

thresholds = np.linspace(0.01, 0.99, 99)
costs = [expected_cost(y_test, proba, t, cost_fp=C_FP, cost_fn=C_FN) for t in thresholds]

best_idx = int(np.argmin(costs))
best_t = float(thresholds[best_idx])

cm_best, p_best, r_best = evaluate_at_threshold(y_test, proba, threshold=best_t)
tn, fp, fn, tp = cm_best.ravel()

print("Chosen costs: C_FP =", C_FP, " | C_FN =", C_FN)
print("Best threshold by expected cost:", round(best_t, 2))
print("Confusion Matrix [[TN, FP], [FN, TP]]:\n", cm_best)
print(f"Precision: {p_best:.3f}  Recall: {r_best:.3f}")
print("FP =", fp, " FN =", fn, "  => Expected cost =", int(costs[best_idx]))


Chosen costs: C_FP = 20  | C_FN = 5000
Best threshold by expected cost: 0.01
Confusion Matrix [[TN, FP], [FN, TP]]:
 [[  0 217]
 [  0  33]]
Precision: 0.132  Recall: 1.000
FP = 217  FN = 0   => Expected cost = 4340
